# Exploratory data analysis and possible feature engineering

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet("data/train_data.parquet")
test_df = pd.read_parquet("data/test_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")
test_df.index = pd.to_numeric(test_df.index, errors="coerce")

In [2]:
train_eda = train_df.copy()
int64_cols = train_eda.select_dtypes(include=['Int64']).columns
train_eda[int64_cols] = train_eda[int64_cols].astype(float)

Exploratory data analysis with packages ydata_profiling and sweetviz

ydata_profiling created too buig unloadable html, so making twi different

In [3]:
from data_profiling import ProfileReport

targets = ['target', 'target_annual_roi']

top_50_features = train_eda.drop(columns=targets, errors='ignore').notna().sum().nlargest(50).index.tolist()

final_cols = top_50_features + targets

train_eda_50 = train_eda[final_cols]

profile = ProfileReport(
    train_eda_50, 
    title="EDA: LendingClub (50 most filled features + targets)",
    explorative=True,
    correlations={
        "auto": {"calculate": True},
        "phi_k": {"calculate": True}
    }
)

profile.to_file("eda/data_profiling_top50.html")

/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/data_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 52/52 [00:08<00:00,  6.20it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
from data_profiling import ProfileReport

profile = ProfileReport(
    train_eda, 
    title="EDA: LendingClub (all features)",
    explorative=True,
    interactions=None,
    correlations={
        "auto": {"calculate": True},
        "phi_k": {"calculate": True},
    }
)

profile.to_file("eda/data_profiling_all.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 95/95 [00:12<00:00,  7.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
import sweetviz as sv

report_sv = sv.analyze(train_eda, target_feat="target")
report_sv.show_html("eda/sweetviz.html", open_browser=False)

                                             |          | [  0%]   00:00 -> (? left)

Report eda/sweetviz.html was generated.


We will handle the high cardinality of the addr_state column by applying a transformation for models that cannot process categorical features. In cases of high correlation, we will evaluate dropping the respective columns. Zero and skewed values do not negatively impact gradient boosted decision tree models, and zero and missing values reflect real-world conditions.

Find possible new features to add with OpenFE

In [6]:
import os
import pandas as pd
from openfe import OpenFE, get_candidate_features
import warnings

warnings.filterwarnings("ignore")

target_cols = ["target", "target_annual_roi"]
X_train = train_df.drop(columns=target_cols)
y_train = train_df["target"]

datetime_cols = X_train.select_dtypes(include=['datetime', 'datetimetz']).columns.tolist()
X_train = X_train.drop(columns=datetime_cols)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

all_candidates = get_candidate_features(
    numerical_features=num_cols,
    categorical_features=cat_cols 
)

allowed_operations = ['/', '*', 'groupby']
filtered_candidates = [
    f for f in all_candidates 
    if any(op in f.name for op in allowed_operations)
]

ofe = OpenFE()
n_cores = max(1, os.cpu_count() - 1)

features_cat = ofe.fit(
    data=X_train.tail(300000), 
    label=y_train.tail(300000), 
    n_jobs=n_cores,
    candidate_features_list=filtered_candidates,
    verbose=False
)

  0%|          | 0/28 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 6705, number of negative: 23295
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000239 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 1
[LightGBM] [Info] Number of positive: 6705, number of negative: 23295
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000199 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 1
[LightGBM] [Info] Number of positive: 6705, number of negative: 23295
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000683 seconds.
You can set `force_row_wise=true` to remove the 

  4%|▎         | 1/28 [00:16<07:12, 16.02s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 6705, number of negative: 23295
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000159 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 1
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 6705, number of negative: 23295
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 1
[LightGBM] [Warning] No further splits with positive gain, be

  7%|▋         | 2/28 [00:16<02:55,  6.74s/it]

...

[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001451 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 1
[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 1
[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001764 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 243
[LightGBM] [I

 96%|█████████▋| 27/28 [03:55<00:04,  4.88s/it]

[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001884 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 1
[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 1
[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001228 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [I

100%|██████████| 28/28 [00:23<00:00,  1.22it/s]


[LightGBM] [Info] Number of positive: 54209, number of negative: 185791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.551854 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 457754
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 2091


In [7]:
def get_formula(node):
    if not hasattr(node, 'children') or not node.children:
        return getattr(node, 'name', str(node))
    
    if len(node.children) == 2:
        left = get_formula(node.children[0])
        right = get_formula(node.children[1])
        if node.name.startswith("GroupByThen"):
            stat_type = node.name.split("Then")[1]
            return f"{stat_type}({left} grouped by {right})"
        return f"({left} {node.name} {right})"
    if len(node.children) == 1:
        return f"{node.name}({get_formula(node.children[0])})"
        
    return node.name

top_n = 100
top_features_cat = features_cat[:top_n]

print(f"\n Totally generated features: {len(features_cat)}")
print(f" Best {top_n} features:\n")

candidate_features = {}

for i, f in enumerate(top_features_cat):
    formula = get_formula(f)
    print(f"Order #{i+1} | Formula: {formula}")
    
    if hasattr(f, 'children') and len(f.children) == 2:
        op = f.name
        if op in ['/', '*']: 
            col1 = f.children[0].name
            col2 = f.children[1].name
            
            op_str = 'div' if op == '/' else 'x'
            feat_name = f"fe_{col1}_{op_str}_{col2}"
            
            candidate_features[feat_name] = (col1, op, col2)


 Totally generated features: 2000
 Best 100 features:

Order #1 | Formula: (sub_grade * term_months)
Order #2 | Formula: (int_rate / fico_avg)
Order #3 | Formula: (int_rate * term_months)
Order #4 | Formula: (sub_grade * dti)
Order #5 | Formula: (fico_avg / term_months)
Order #6 | Formula: (sub_grade * num_actv_rev_tl)
Order #7 | Formula: (acc_open_past_24mths / tot_hi_cred_lim)
Order #8 | Formula: (sub_grade / avg_cur_bal)
Order #9 | Formula: (loan_amnt / annual_inc)
Order #10 | Formula: (total_rev_hi_lim / num_rev_tl_bal_gt_0)
Order #11 | Formula: (dti / mort_acc)
Order #12 | Formula: (dti / avg_cur_bal)
Order #13 | Formula: (installment / avg_cur_bal)
Order #14 | Formula: (total_rev_hi_lim / num_actv_rev_tl)
Order #15 | Formula: (acc_open_past_24mths / mort_acc)
Order #16 | Formula: (sub_grade * acc_open_past_24mths)
Order #17 | Formula: (loan_amnt * sub_grade)
Order #18 | Formula: (annual_inc / num_actv_rev_tl)
Order #19 | Formula: (installment * dti)
Order #20 | Formula: (int_rat

In [8]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit, cross_validate

def safe_divide(df, col_num, col_den):
    return df[col_num] / df[col_den].replace(0, np.nan)

def evaluate_cv(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    clf = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
    results = cross_validate(clf, X, y, cv=tscv, scoring='roc_auc')
    return np.mean(results['test_score'])


current_best_auc = evaluate_cv(X_train, y_train)
print(f"Baseline Val AUC: {current_best_auc:.5f}\n")

X_best = X_train.copy()
accepted_features = []

for feat_name, (col1, op, col2) in candidate_features.items():
    
    X_temp = X_best.copy()
        
    if op == '/':
        X_temp[feat_name] = safe_divide(X_temp, col1, col2)
    elif op == '*':
        X_temp[feat_name] = X_temp[col1] * X_temp[col2]

    new_auc = evaluate_cv(X_temp, y_train)
    
    improvement = new_auc - current_best_auc
    
    if improvement > 0.0001:
        print(f"ACCEPTED: {feat_name} | AUC raised by: {improvement:.5f} (New: {new_auc:.5f})")
        X_best = X_temp.copy()
        current_best_auc = new_auc
        accepted_features.append(feat_name)
    else:
        print(f"REJECTED: {feat_name} | AUC changed by: {improvement:.5f}")


print(f"Used {len(accepted_features)} Features:")
for f in accepted_features:
    print(f" - {f}")

Baseline Val AUC: 0.73040

ACCEPTED: fe_sub_grade_x_term_months | AUC raised by: 0.00019 (New: 0.73060)
ACCEPTED: fe_int_rate_div_fico_avg | AUC raised by: 0.00011 (New: 0.73070)
REJECTED: fe_int_rate_x_term_months | AUC changed by: -0.00050
REJECTED: fe_sub_grade_x_dti | AUC changed by: -0.00021
REJECTED: fe_fico_avg_div_term_months | AUC changed by: -0.00004
REJECTED: fe_sub_grade_x_num_actv_rev_tl | AUC changed by: -0.00017
REJECTED: fe_acc_open_past_24mths_div_tot_hi_cred_lim | AUC changed by: -0.00011
REJECTED: fe_sub_grade_div_avg_cur_bal | AUC changed by: -0.00051
REJECTED: fe_loan_amnt_div_annual_inc | AUC changed by: -0.00066
REJECTED: fe_total_rev_hi_lim_div_num_rev_tl_bal_gt_0 | AUC changed by: 0.00002
REJECTED: fe_dti_div_mort_acc | AUC changed by: -0.00007
REJECTED: fe_dti_div_avg_cur_bal | AUC changed by: -0.00033
REJECTED: fe_installment_div_avg_cur_bal | AUC changed by: -0.00026
REJECTED: fe_total_rev_hi_lim_div_num_actv_rev_tl | AUC changed by: 0.00009
REJECTED: fe_acc

Features are not useful and if yes by very small margin so we will use only existing features which are already 92 + 2 target columns.

In [ ]:
"""
train_df_fe = train_df.copy()
test_df_fe = test_df.copy()

for feat_name in accepted_features:
    col1, op, col2 = candidate_features[feat_name]
    
    if op == '/':
        train_df_fe[feat_name] = safe_divide(train_df_fe, col1, col2)
        test_df_fe[feat_name] = safe_divide(test_df_fe, col1, col2)
    elif op == '*':
        train_df_fe[feat_name] = train_df_fe[col1] * train_df_fe[col2]
        test_df_fe[feat_name] = test_df_fe[col1] * test_df_fe[col2]
"""

Look for possible improvement if we drop existing features

In [10]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit, cross_validate

def evaluate_cv(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    clf = lgb.LGBMClassifier(
        random_state=42, 
        n_jobs=-1, 
        verbose=-1,
        subsample=1.0,
        colsample_bytree=1.0
    )
    results = cross_validate(clf, X, y, cv=tscv, scoring='roc_auc')
    return np.mean(results['test_score'])

X_best = X_best.drop(columns=accepted_features)

current_best_auc = evaluate_cv(X_best, y_train)
print(f"Baseline Val AUC: {current_best_auc:.5f}\n")

clf_imp = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, subsample=1.0, colsample_bytree=1.0)
clf_imp.fit(X_best, y_train)

importances = pd.Series(clf_imp.feature_importances_, index=X_best.columns).sort_values()
features_to_test = importances.index.tolist()

dropped_features = []

for feat_name in features_to_test:
    
    X_temp = X_best.drop(columns=[feat_name])
    
    new_auc = evaluate_cv(X_temp, y_train)
    
    improvement = new_auc - current_best_auc
    
    if improvement > 0.00005: 
        print(f"DELETED: {feat_name} | AUC improved by {improvement:.5f} (New: {new_auc:.5f})")
        X_best = X_temp.copy()
        current_best_auc = new_auc
        dropped_features.append(feat_name)
    else:
        print(f"KEPT: {feat_name} | AUC changed by: {improvement:.5f}")

print(f"Final Val AUC: {current_best_auc:.5f}")
print(f"From the original {X_best.shape[1]} features, {len(dropped_features)} were deleted.") 

Baseline Val AUC: 0.73039

KEPT: earliest_cr_line_month | AUC changed by: -0.00002
KEPT: open_il_12m | AUC changed by: 0.00000
KEPT: open_il_24m | AUC changed by: 0.00000
KEPT: tax_liens | AUC changed by: -0.00003
KEPT: pub_rec | AUC changed by: -0.00002
DELETED: open_acc | AUC improved by 0.00012 (New: 0.73051)
KEPT: acc_now_delinq | AUC changed by: 0.00000
KEPT: chargeoff_within_12_mths | AUC changed by: 0.00001
KEPT: delinq_amnt | AUC changed by: 0.00002
KEPT: earliest_cr_line_month_cos | AUC changed by: -0.00007
KEPT: issue_d_month_sin | AUC changed by: -0.00028
KEPT: disbursement_method | AUC changed by: 0.00000
KEPT: num_op_rev_tl | AUC changed by: -0.00016
KEPT: num_tl_30dpd | AUC changed by: -0.00005
KEPT: num_tl_90g_dpd_24m | AUC changed by: -0.00017
KEPT: application_type | AUC changed by: 0.00000
KEPT: num_accts_ever_120_pd | AUC changed by: -0.00014
KEPT: earliest_cr_line_month_sin | AUC changed by: -0.00006
KEPT: inq_last_12m | AUC changed by: 0.00000
KEPT: num_sats | AUC 

No significant improvement when dropping so keeping all features and not changing the dataset

In [ ]:
"""
train_df_fe = train_df_fe.drop(columns=dropped_features, errors='ignore')
test_df_fe = test_df_fe.drop(columns=dropped_features, errors='ignore')

train_df_fe.index = train_df_fe.index.astype(str)
test_df_fe.index = test_df_fe.index.astype(str)

train_df_fe.to_parquet("data/train_data_fe.parquet", index=False)
test_df_fe.to_parquet("data/test_data_fe.parquet", index=False)

train_df_fe.index = pd.to_numeric(train_df_fe.index, errors="coerce")
test_df_fe.index = pd.to_numeric(test_df_fe.index, errors="coerce")
"""